# Task 3: Model Explainability and Business Insights

This notebook focuses on interpreting the fraud detection model using **built-in feature importance** and **SHAP (SHapley Additive exPlanations)**.

## Objectives:
1.  **Built-in Feature Importance**: Identify the top 10 features directly from the ensemble model.
2.  **Global SHAP Analysis**: Understand the overall feature impact across the dataset.
3.  **Local SHAP Analysis**: Explain specific model decisions for:
    - One **True Positive** (Correctly flagged fraud)
    - One **False Positive** (Incorrectly flagged legitimate)
    - One **False Negative** (Missed fraud)
4.  **Comparison**: Contrast SHAP findings with built-in importance.
5.  **Business Recommendations**: Provide actionable insights linked to model findings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path
sys.path.append('../src')

from preprocessor import Preprocessor
from modeling import ModelTrainer
from explainability import ModelExplainer

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data and Prepare Model

We re-train our selected **Random Forest** model on the e-commerce dataset.

In [ ]:
# Load data
fraud_df = pd.read_csv('../data/processed/fraud_data_engineered.csv')

# Preprocess
preprocessor = Preprocessor()
X, y = preprocessor.prepare_for_modeling(fraud_df, target_col='class', handle_imbalance=False)
X_train, X_test, y_train, y_test = preprocessor.stratified_split(X, y, test_size=0.2)

# Train Random Forest
trainer = ModelTrainer(random_state=42)
model = trainer.train_ensemble_random_forest(X_train, y_train, tune=False, n_estimators=100)

print(f"Model trained on {X_train.shape[0]} samples with {X_train.shape[1]} features.")

## 2. Built-in Feature Importance

We extract the importance values directly from the Random Forest model and visualize the top 10 features.

In [ ]:
# Extract and plot top 10 built-in feature importances
importances = model.feature_importances_
indices = np.argsort(importances)[::-1][:10]
feature_names = X.columns

plt.figure(figsize=(10, 6))
plt.title("Top 10 Built-in Feature Importances (Random Forest)")
sns.barplot(x=importances[indices], y=feature_names[indices], palette='viridis')
plt.xlabel("Gini Importance")
plt.show()

## 3. SHAP Analysis

### 3.1 Initialize SHAP Explainer

In [ ]:
# Initialize the explainer
explainer_obj = ModelExplainer(model, X_train)

# Use a subset for SHAP computation for speed
X_explain = X_test.head(500)
y_explain = y_test.head(500)

### 3.2 Global Feature Importance (SHAP Summary Plot)

This plot shows the impact of feature values on the model output across the subset.

In [ ]:
explainer_obj.plot_summary(X_explain)

### 3.3 Local Interpretability: Three Specific Cases

We identify and explain one True Positive, one False Positive, and one False Negative.

In [ ]:
# Get predictions for finding examples
y_pred = model.predict(X_explain)

# Find indices for different cases
examples = explainer_obj.find_prediction_examples(X_explain, y_explain, y_pred)
print(f"Found examples: {examples}")

#### Case 1: True Positive (Correctly Flagged Fraud)

In [ ]:
if 'true_positive' in examples:
    print("Explaining True Positive case...")
    explainer_obj.plot_waterfall(X_explain, examples['true_positive'])
else:
    print("No True Positive found in subset.")

#### Case 2: False Positive (Legitimate Flagged as Fraud)

In [ ]:
if 'false_positive' in examples:
    print("Explaining False Positive case...")
    explainer_obj.plot_waterfall(X_explain, examples['false_positive'])
else:
    print("No False Positive found in subset.")

#### Case 3: False Negative (Fraud Missed by Model)

In [ ]:
if 'false_negative' in examples:
    print("Explaining False Negative case...")
    explainer_obj.plot_waterfall(X_explain, examples['false_negative'])
else:
    print("No False Negative found in subset.")

## 4. Interpretation and Comparison

### 4.1 SHAP vs Built-in Importance

We compare the rankings from both methods to see if they align.

In [ ]:
shap_importance = explainer_obj.get_top_features(X_explain)
print("SHAP Top Features:")
print(shap_importance.head(10))

print("\nBuilt-in Top 10 Features:")
print(feature_names[indices])

### 4.2 Top 5 Drivers of Fraud

Based on the analysis, the top 5 drivers are:
1.  **time_since_signup**: New accounts are the single strongest indicator of fraud. Transactions occurring almost immediately after signup are highly suspicious.
2.  **purchase_value**: High transaction amounts increase risk, as fraudsters often aim for maximum gain before detection.
3.  **hour/purchase_hour**: Fraud patterns vary by time of day, often peaking during off-peak hours when human monitoring might be lower.
4.  **age**: Certain age ranges show higher vulnerability or are targeted by specific fraud schemes.
5.  **source/browser**: Specific acquisition channels or technical setups are preferred by automated fraud scripts.

**Unexpected Outcomes**: We noticed that while `country` was expected to be a top driver, its impact was localized to specific regions, often outweighed by temporal and behavioral patterns like signup velocity.

## 5. Business Recommendations

### Actionable Insights for Adey Innovations Inc.

1.  🔍 **Enhanced Verification for New Users**: Since `time_since_signup` is critical, implement mandatory multi-factor authentication (MFA) or manual review for any transaction occurring within the first 12 hours of account creation. This directly targets the highest impact feature.

2.  💰 **Dynamic Thresholds for Large Transactions**: Based on the importance of `purchase_value`, set adaptive limits that trigger additional security steps (e.g., identity verification) for transactions that are significantly above the user's historical average or the platform mean for their demographic.

3.  ⏰ **Time-Adaptive Risk Scoring**: Leverage temporal findings (hour of transaction) to implement stricter automated blocking rules during high-risk windows (e.g., 1 AM - 5 AM). This compensates for potential gaps in human surveillance and addresses the behavioral patterns identified by the model.

4.  📱 **Device & Connection Fingerprinting**: Link the behavioral drivers to device monitoring. If a high-value transaction originates from a new device for a young account (`age` + `time_since_signup`), escalate the risk level to 'Critical' immediately.